# Lab Solutions: Deep RL with LunarLander

This notebook contains the complete lab code with expected results and additional insights.

---

In [ ]:
!apt-get install -y swig
!pip install stable-baselines3[extra] gymnasium[box2d] -q

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback
import time

## Task 0 Solution: CartPole, the Library Way

PPO with default settings usually solves CartPole (500) within 50k steps - steadier
than the hand-written REINFORCE from week 3, which got there with dips along the way.

In [ ]:
cartpole = gym.make('CartPole-v1')
model = PPO('MlpPolicy', cartpole, verbose=0, seed=42)
model.learn(total_timesteps=50_000)

mean_cp, std_cp = evaluate_policy(model, cartpole, n_eval_episodes=100)
print(f"CartPole with PPO: {mean_cp:.1f} +/- {std_cp:.1f}   (500 is the maximum)")

## Task 1 Solution: Exploring the Environment

**Expected output:**
- Observation space: Box(8,)
- Action space: Discrete(4)
- Random policy mean reward: ~-150 to -250 (crashes!)

In [ ]:
env = gym.make('LunarLander-v3')

print("Environment Info:")
print(f"  Observation space: {env.observation_space}")
print(f"  Action space: {env.action_space}")

state, _ = env.reset(seed=42)
print(f"\nInitial state: {state}")


def evaluate_random_policy(env, n_episodes=10):
    episode_rewards = []
    for episode in range(n_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            action = env.action_space.sample()
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        
        episode_rewards.append(total_reward)
    return episode_rewards


random_rewards = evaluate_random_policy(env, n_episodes=10)
print(f"\nRandom Policy Performance:")
print(f"  Mean: {np.mean(random_rewards):.2f}")
print(f"  Std: {np.std(random_rewards):.2f}")
print(f"\n✓ Expected: Mean around -150 to -250 (random agents crash)")

## Task 2 Solution: Training PPO

**What to expect after 100k steps:**
- Clearly better than random - the agent has stopped simply falling out of the sky
- Usually **not** solved yet: 100k steps is a short budget for LunarLander, and the
  score varies a lot from seed to seed (in our test runs, anywhere from about -70 to +60)
- Training time: a few minutes on Colab's CPU

Reaching 200 reliably typically takes several hundred thousand to around a million
steps, or tuned hyperparameters (see the [RL Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)).

In [ ]:
env = gym.make('LunarLander-v3')

model_ppo = PPO(
    'MlpPolicy',
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1,
    seed=42
)

print("Training PPO...\n")
start = time.time()
model_ppo.learn(total_timesteps=100_000)
train_time = time.time() - start

mean_reward, std_reward = evaluate_policy(model_ppo, env, n_eval_episodes=100)

print(f"\n{'='*50}")
print(f"PPO Results:")
print(f"  Training time: {train_time:.1f}s")
print(f"  Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"  Solved: {'✓' if mean_reward >= 200 else '✗'}")
print(f"{'='*50}")
print(f"\nRandom policy scored {np.mean(random_rewards):.0f} - the gap to that is what 100k steps bought.")
if mean_reward >= 200:
    print("Solved already - a lucky seed at this budget.")
else:
    print("Not solved yet, which is normal at 100k steps. Task 5 trains longer.")

## Task 3 Solution: Algorithm Comparison

**What to expect (100k steps, one seed each):**
- All three should be well above random, and usually all three are still short of 200
- The ranking is **not stable**: it changes with the seed. In our own test runs A2C
  came out ahead of PPO on average at this budget, and DQN varied the most
- PPO tends to be the more dependable choice over longer budgets and across many
  environments - which is why it is a common default - but a short single-seed run
  will not necessarily show that

This is the honest lesson of the comparison: **one seed and one budget cannot rank
algorithms.** To claim "X beats Y" you would need several seeds per algorithm and a
look at the spread.

In [ ]:
# Train A2C (same settings as the lab)
print("Training A2C...\n")
env = gym.make('LunarLander-v3')
model_a2c = A2C('MlpPolicy', env, learning_rate=7e-4, n_steps=5, gamma=0.99,
                verbose=1, seed=42)
model_a2c.learn(total_timesteps=100_000)
mean_a2c, std_a2c = evaluate_policy(model_a2c, env, n_eval_episodes=100)
print(f"A2C: {mean_a2c:.2f} ± {std_a2c:.2f}")

In [ ]:
# Train DQN (same settings as the lab)
print("\nTraining DQN...\n")
env = gym.make('LunarLander-v3')
model_dqn = DQN('MlpPolicy', env, learning_rate=1e-4, buffer_size=50000,
                learning_starts=1000, batch_size=64, gamma=0.99,
                verbose=1, seed=42)
model_dqn.learn(total_timesteps=100_000)
mean_dqn, std_dqn = evaluate_policy(model_dqn, env, n_eval_episodes=100)
print(f"DQN: {mean_dqn:.2f} ± {std_dqn:.2f}")

In [ ]:
# Comparison plot
algorithms = ['Random', 'PPO', 'A2C', 'DQN']
mean_rewards = [np.mean(random_rewards), mean_reward, mean_a2c, mean_dqn]
std_rewards = [np.std(random_rewards), std_reward, std_a2c, std_dqn]

plt.figure(figsize=(10, 6))
colors = ['red', 'blue', 'green', 'orange']
bars = plt.bar(algorithms, mean_rewards, yerr=std_rewards, 
               capsize=5, color=colors, alpha=0.7)
plt.axhline(y=200, color='black', linestyle='--', label='Solved (200)')
plt.ylabel('Mean Reward')
plt.title('Algorithm Comparison')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

for bar, mean in zip(bars, mean_rewards):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{mean:.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

ranked = sorted(zip(algorithms[1:], mean_rewards[1:]), key=lambda x: -x[1])
print("\nRanking in this run:", " > ".join(f"{a} ({m:.0f})" for a, m in ranked))
print("\nTypical observations:")
print("  - All three are far above random, none is reliably at 200 yet")
print("  - The order changes from seed to seed at this budget - do not read")
print("    a general ranking into one run")
print("  - Note the error bars: within one agent, episodes still vary a lot")

## Task 4 Solution: Hyperparameter Experiments

**What to expect at 50k steps:**
- None of the three gets close to 200 - 50k steps is very short
- A short budget tends to **favour a higher learning rate**: it simply gets further in
  the steps available. In our test runs 1e-3 did best and 1e-4 did clearly worst
  (sometimes far worse than random - an agent that fires its engines all the time
  racks up fuel costs)
- The default 3e-4 is a sensible middle ground for longer runs, where a high rate
  is more likely to become unstable

**Key insight:** the "best" learning rate depends on the training budget. A
hyperparameter comparison is only meaningful for the budget it was run at.

In [ ]:
def train_and_evaluate(learning_rate, timesteps=50_000):
    env = gym.make('LunarLander-v3')
    model = PPO('MlpPolicy', env, learning_rate=learning_rate, 
                verbose=0, seed=42)
    model.learn(total_timesteps=timesteps)
    mean, std = evaluate_policy(model, env, n_eval_episodes=50)
    return mean, std


learning_rates = [1e-4, 3e-4, 1e-3]
results = {}

print("Testing different learning rates...\n")
for lr in learning_rates:
    print(f"LR={lr:.0e}: ", end="")
    mean, std = train_and_evaluate(lr)
    results[lr] = (mean, std)
    print(f"{mean:.2f} ± {std:.2f}")

# Plot
plt.figure(figsize=(10, 6))
lrs = [f"{lr:.0e}" for lr in learning_rates]
means = [results[lr][0] for lr in learning_rates]
stds = [results[lr][1] for lr in learning_rates]

plt.bar(lrs, means, yerr=stds, capsize=5, alpha=0.7, color='steelblue')
plt.axhline(y=200, color='g', linestyle='--', label='Solved')
plt.xlabel('Learning Rate')
plt.ylabel('Mean Reward')
plt.title('Impact of Learning Rate (50k steps)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

best_lr = max(results, key=lambda lr: results[lr][0])
print(f"\nBest at 50k steps in this run: LR={best_lr:.0e}")
print("Typical patterns:")
print("  - Nothing reaches 200 in 50k steps")
print("  - Short budgets tend to reward faster learning (a higher LR)")
print("  - Whether that still holds at 300k+ steps is a different question")

## Task 5 Solution (Optional): Longer Training

**What to expect after 300k steps:**
- Clearly better than at 100k steps (around 130 in our test run)
- Often still not solved - and the evaluation curve is **not** monotonic: it can dip
  between checkpoints
- That is what the `EvalCallback` is for: it keeps the best checkpoint in
  `./logs/best_model.zip`, which may well be better than the final model

In [ ]:
eval_env = gym.make('LunarLander-v3')
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path='./logs/',
    log_path='./logs/',
    eval_freq=5000,
    n_eval_episodes=20,
    deterministic=True
)

env = gym.make('LunarLander-v3')
model_long = PPO('MlpPolicy', env, verbose=1, seed=42)

print("Training for 300k steps...\n")
model_long.learn(total_timesteps=300_000, callback=eval_callback)

mean_long, std_long = evaluate_policy(model_long, env, n_eval_episodes=100)
print(f"\nFinal: {mean_long:.2f} ± {std_long:.2f}")
print(f"\nCompared with 100k steps: {mean_reward:.2f} -> {mean_long:.2f}")

# The best checkpoint seen during training is often better than the final one
evals = np.load('./logs/evaluations.npz')
best_i = evals['results'].mean(axis=1).argmax()
print(f"Best evaluation during training: {evals['results'][best_i].mean():.2f} "
      f"at {evals['timesteps'][best_i]:,} steps (saved in ./logs/best_model.zip)")

---

## Additional Insights

### Why PPO Is a Common Default

1. **Clipped objective**: Prevents too-large policy updates
2. **Multiple epochs**: Reuses data efficiently
3. **Forgiving defaults**: Tends to work reasonably across many environments without much tuning

It does not win everywhere - and with short budgets, as here, it may not win at all.

### When to Use Each Algorithm

- **PPO**: Default choice, most reliable
- **A2C**: When you need speed and can tolerate variance
- **DQN**: Discrete actions, when you have good hyperparameters

### Common Issues & Solutions

**Problem: Agent doesn't learn (stuck at -200)**
- Check learning rate (try 3e-4)
- Train longer (100k → 300k steps)
- Try different random seed

**Problem: High variance in performance**
- Increase batch size (64 → 128)
- Use more evaluation episodes (20 → 100)
- Train longer for stability

**Problem: Training is too slow**
- Reduce timesteps for experiments (100k → 50k)
- Use fewer evaluation episodes
- Try A2C (faster but less stable)

### Hyperparameter Cheat Sheet

**Conservative (stable but slow):**
```python
PPO(..., learning_rate=1e-4, n_steps=2048, batch_size=128)
```

**Balanced (recommended):**
```python
PPO(..., learning_rate=3e-4, n_steps=2048, batch_size=64)
```

**Aggressive (fast but risky):**
```python
PPO(..., learning_rate=1e-3, n_steps=1024, batch_size=32)
```

---

## Summary of What to Expect

| Task | Algorithm | Steps | What you will typically see |
|------|-----------|-------|-----------------------------|
| 0 | PPO on CartPole | 50k | Solved, or very close |
| 1 | Random | - | Around -200: it crashes |
| 2 | PPO | 100k | Far above random, usually not solved |
| 3 | A2C, DQN | 100k | Same picture; ranking changes with the seed |
| 4 | PPO, three LRs | 50k | None near 200; short budgets favour higher LRs |
| 5 | PPO | 300k | Clearly better again, often still short of 200 |

**Remember:** single runs vary a lot with the random seed and library versions.
The shape of the results matters more than any particular number. 🚀